In [1]:
!pip install -q --upgrade google-cloud-texttospeech

In [2]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "focal-pathway-436901-j2-6aa3227a5b44.json"


In [3]:
!pip install tensorflow[and-cuda]

In [4]:
!pip install keras-nlp

In [ ]:
#NOT NEEDED CAN REMOVE
!pip install --upgrade --quiet google-cloud-aiplatform

In [ ]:
#NOT NEEDED CAN REMOVE
from google.cloud import aiplatform

### install gdown get the model file from drive


install gdown get the model file from drive

In [ ]:
!pip install gdown


In [1]:
#start off running this
# import keras so it is defined for later use
import keras
import keras_nlp

# Load the saved model in .keras format
gemma_lm = keras.models.load_model('my_model.keras')

2024-11-17 23:19:41.840050: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-11-17 23:19:42.399559: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1731885582.592058    3443 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1731885582.646668    3443 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-11-17 23:19:43.156571: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [ ]:
# Install Gradio
!pip install gradio


In [3]:
import gradio as gr
import tensorflow as tf
import keras
import speech_recognition as sr
from google.cloud import texttospeech

# Initialize the Text-to-Speech client
client = texttospeech.TextToSpeechClient()

# Initialize the Speech Recognition
recognizer = sr.Recognizer()

# Function to process Speech-to-Text
def speech_to_text(audio):
    try:
        with sr.AudioFile(audio) as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data)
            return text
    except sr.UnknownValueError:
        return "Sorry, I couldn't understand the audio. Please try again."
    except sr.RequestError as e:
        return f"Request error from Google STT service; {e}"

# Function to generate response from the chatbot model
def generate_response(model: keras.Model, question: str, max_length: int = 250) -> str:
    prompt_template = """ Role: You are a helpful real estate assistant providing support to legal matters in Ontario, 
    Canada Region; always answer the questions based on the following instructions; given the question, 
    generate an answer in 5 to 6 sentences always. For every statement in the answer, provide supporting 
    legal document or law segments from Ontario, Canada region.
    Question:\n{question}\n\nAnswer:\n"""
    
    prompt = prompt_template.format(question=question)
    
    try:
        generated_response = model.generate(prompt, max_length=max_length)
        generated_text = generated_response.decode('utf-8') if isinstance(generated_response, bytes) else generated_response
        answer_text = generated_text[len(prompt):].strip()
        return answer_text
    except Exception as e:
        return f"⚠️ Error generating answer: {e}"

# Function to synthesize text using Google Text-to-Speech
def synthesize_text(text):
    input_text = texttospeech.SynthesisInput(text=text)
    voice = texttospeech.VoiceSelectionParams(
        language_code="en-US",
        name="en-US-Standard-C",
        ssml_gender=texttospeech.SsmlVoiceGender.FEMALE
    )
    audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)
    response = client.synthesize_speech(
        request={"input": input_text, "voice": voice, "audio_config": audio_config}
    )
    audio_file_path = "output.mp3"
    with open(audio_file_path, "wb") as out:
        out.write(response.audio_content)
    return audio_file_path

# Combined function to handle chatbot response and TTS
def chatbot_with_tts_and_stt(audio):
    # Convert speech to text
    user_input = speech_to_text(audio)
    if not user_input.strip():
        return "Chatbot: I'm here to help! Please try speaking again.", None
    
    # Confirm the input text
    confirmation_message = f"Recognized text: '{user_input}'. Is this correct? If so, proceed. If not, try again."
    # Optionally, you can add a confirmation mechanism here
    
    # Generate chatbot response
    answer = generate_response(gemma_lm, user_input, max_length=5000)
    
    # Synthesize the generated text into speech
    audio_file_path = synthesize_text(answer)
    
    return answer, audio_file_path

# Create Gradio interface
interface = gr.Interface(
    fn=chatbot_with_tts_and_stt,
    inputs=gr.Audio(sources=["microphone"], type="filepath"),
    outputs=[gr.Textbox(), gr.Audio(autoplay=True)],
    title="Chatbot with STT and TTS",
    description="Speak your question and get a spoken answer."
)

# Launch the interface
interface.launch(share=True)



* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://37d5ba709a8d1f067c.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


2024-11-17 23:23:11.257202: E tensorflow/core/util/util.cc:131] oneDNN supports DT_INT64 only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
I0000 00:00:1731885798.170604    4296 service.cc:148] XLA service 0x7fae43433f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1731885798.172284    4296 service.cc:156]   StreamExecutor device (0): NVIDIA L4, Compute Capability 8.9
2024-11-17 23:23:19.902623: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1731885802.006262    4296 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1731885809.067429    4296 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
